In [ ]:
%%sql -r dataframe_2
use  food_delivery_db

In [ ]:
%%sql -r dataframe_5
TRUNCATE TABLE GOLD.DIM_DATE;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE GOLD.DIM_DATE (

    DATE_SK NUMBER PRIMARY KEY,
    FULL_DATE DATE,
    DAY_NUMBER NUMBER,
    MONTH_NUMBER NUMBER,
    MONTH_NAME VARCHAR,
    QUARTER_NUMBER NUMBER,
    YEAR_NUMBER NUMBER,
    WEEKDAY_NAME VARCHAR,
    IS_WEEKEND BOOLEAN

);

In [ ]:
%%sql -r dataframe_3
INSERT INTO GOLD.DIM_DATE
SELECT DISTINCT

    TO_NUMBER(TO_CHAR(CAST(ORDER_PLACED_AT AS DATE), 'YYYYMMDD')) AS DATE_SK,

    CAST(ORDER_PLACED_AT AS DATE) AS FULL_DATE,

    DAY(CAST(ORDER_PLACED_AT AS DATE)) AS DAY_NUMBER,

    MONTH(CAST(ORDER_PLACED_AT AS DATE)) AS MONTH_NUMBER,

    MONTHNAME(CAST(ORDER_PLACED_AT AS DATE)) AS MONTH_NAME,

    QUARTER(CAST(ORDER_PLACED_AT AS DATE)) AS QUARTER_NUMBER,

    YEAR(CAST(ORDER_PLACED_AT AS DATE)) AS YEAR_NUMBER,

    DAYNAME(CAST(ORDER_PLACED_AT AS DATE)) AS WEEKDAY_NAME,

    CASE
        WHEN DAYNAME(CAST(ORDER_PLACED_AT AS DATE)) IN ('Sat', 'Sun')
        THEN TRUE
        ELSE FALSE
    END AS IS_WEEKEND

FROM SILVER.ORDER_CLEAN;

In [ ]:
%%sql -r dataframe_4
SELECT *
FROM GOLD.DIM_DATE
ORDER BY FULL_DATE
LIMIT 10;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE GOLD.DIM_AGENT AS
SELECT

    ROW_NUMBER() OVER (ORDER BY AGENT_ID) AS AGENT_SK,

    AGENT_ID,

    AGENT_NAME,

    PHONE_NUMBER,

    CITY,

    VEHICLE_TYPE,

    JOINING_DATE,

    AGENT_RATING,

    AVAILABILITY_STATUS

FROM SILVER.AGENT_CLEAN;

In [ ]:
%%sql -r dataframe_7
SELECT *
FROM GOLD.DIM_AGENT
LIMIT 20;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE GOLD.DIM_PROMOTION AS
SELECT

    ROW_NUMBER() OVER (ORDER BY PROMO_CODE) AS PROMOTION_SK,

    PROMO_CODE,

    PROMO_TYPE,

    DISCOUNT_VALUE,

    MIN_ORDER_VALUE,

    START_DATE,

    END_DATE,

    IS_ACTIVE

FROM SILVER.PROMOTION_CLEAN;

In [ ]:
%%sql -r dataframe_9
SELECT *
FROM GOLD.DIM_PROMOTION
LIMIT 20;

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE TABLE GOLD.DIM_RESTAURANT AS
SELECT

    ROW_NUMBER() OVER (ORDER BY RESTAURANT_ID) AS RESTAURANT_SK,

    RESTAURANT_ID,
    RESTAURANT_NAME,
    CUISINE_TYPE,
    CITY,
    STATE,
    PINCODE,
    RATING,
    AVERAGE_PREP_TIME,
    COMMISSION_RATE,
    OPENING_TIME,
    CLOSING_TIME,
    IS_ACTIVE

FROM SILVER.RESTAURANT_CLEAN;

In [ ]:
%%sql -r dataframe_11
SELECT *
FROM GOLD.DIM_RESTAURANT
LIMIT 20;

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE TABLE GOLD.DIM_CUSTOMER (

    CUSTOMER_SK NUMBER AUTOINCREMENT,

    CUSTOMER_ID VARCHAR,

    FIRST_NAME VARCHAR,
    LAST_NAME VARCHAR,
    EMAIL VARCHAR,
    PHONE_NUMBER VARCHAR,
    DATE_OF_BIRTH DATE,
    GENDER VARCHAR,
    ADDRESS_LINE1 VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR,
    PINCODE VARCHAR,
    SIGNUP_DATE DATE,
    CUSTOMER_SEGMENT VARCHAR,
    IS_ACTIVE BOOLEAN,

    EFFECTIVE_START_DATE DATE,
    EFFECTIVE_END_DATE DATE,
    IS_CURRENT BOOLEAN
);

In [ ]:
%%sql -r dataframe_13
INSERT INTO GOLD.DIM_CUSTOMER (

    CUSTOMER_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    DATE_OF_BIRTH,
    GENDER,
    ADDRESS_LINE1,
    CITY,
    STATE,
    PINCODE,
    SIGNUP_DATE,
    CUSTOMER_SEGMENT,
    IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    IS_CURRENT
)

SELECT

    CUSTOMER_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    DATE_OF_BIRTH,
    GENDER,
    ADDRESS_LINE1,
    CITY,
    STATE,
    PINCODE,
    SIGNUP_DATE,
    CUSTOMER_SEGMENT,
    IS_ACTIVE,

    CURRENT_DATE(),
    '9999-12-31',
    TRUE

FROM SILVER.CUSTOMER_CLEAN;

In [ ]:
%%sql -r dataframe_14
UPDATE GOLD.DIM_CUSTOMER d
SET
    IS_CURRENT = FALSE,
    EFFECTIVE_END_DATE = CURRENT_DATE() - 1

FROM SILVER.CUSTOMER_CLEAN s

WHERE d.CUSTOMER_ID = s.CUSTOMER_ID
  AND d.IS_CURRENT = TRUE

  AND (
      d.EMAIL <> s.EMAIL
      OR d.PHONE_NUMBER <> s.PHONE_NUMBER
      OR d.ADDRESS_LINE1 <> s.ADDRESS_LINE1
      OR d.CITY <> s.CITY
      OR d.STATE <> s.STATE
      OR d.PINCODE <> s.PINCODE
      OR d.CUSTOMER_SEGMENT <> s.CUSTOMER_SEGMENT
      OR d.IS_ACTIVE <> s.IS_ACTIVE
  );

In [ ]:
%%sql -r dim_customer_insert
INSERT INTO GOLD.DIM_CUSTOMER (

    CUSTOMER_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    DATE_OF_BIRTH,
    GENDER,
    ADDRESS_LINE1,
    CITY,
    STATE,
    PINCODE,
    SIGNUP_DATE,
    CUSTOMER_SEGMENT,
    IS_ACTIVE,
    EFFECTIVE_START_DATE,
    EFFECTIVE_END_DATE,
    IS_CURRENT
)

SELECT

    s.CUSTOMER_ID,
    s.FIRST_NAME,
    s.LAST_NAME,
    s.EMAIL,
    s.PHONE_NUMBER,
    s.DATE_OF_BIRTH,
    s.GENDER,
    s.ADDRESS_LINE1,
    s.CITY,
    s.STATE,
    s.PINCODE,
    s.SIGNUP_DATE,
    s.CUSTOMER_SEGMENT,
    s.IS_ACTIVE,

    CURRENT_DATE(),
    '9999-12-31',
    TRUE

FROM SILVER.CUSTOMER_CLEAN s

LEFT JOIN GOLD.DIM_CUSTOMER d
    ON s.CUSTOMER_ID = d.CUSTOMER_ID
   AND d.IS_CURRENT = TRUE

WHERE

    d.CUSTOMER_ID IS NULL

    OR (
        d.EMAIL <> s.EMAIL
        OR d.PHONE_NUMBER <> s.PHONE_NUMBER
        OR d.ADDRESS_LINE1 <> s.ADDRESS_LINE1
        OR d.CITY <> s.CITY
        OR d.STATE <> s.STATE
        OR d.PINCODE <> s.PINCODE
        OR d.CUSTOMER_SEGMENT <> s.CUSTOMER_SEGMENT
        OR d.IS_ACTIVE <> s.IS_ACTIVE
    );

In [ ]:
%%sql -r dataframe_16
UPDATE SILVER.CUSTOMER_CLEAN
SET CITY = 'Bangalore',
    CUSTOMER_SEGMENT = 'PREMIUM'
WHERE CUSTOMER_ID = 'CUST000002';

In [ ]:
%%sql -r dataframe_15
select * from GOLD.dim_customer where customer_id='CUST000002'

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE TABLE GOLD.FACT_ORDERS AS
SELECT

    o.ORDER_ID,

    dc.CUSTOMER_SK,
    dr.RESTAURANT_SK,
    da.AGENT_SK,
    dp.PROMOTION_SK,
    dd.DATE_SK,

    o.ORDER_STATUS,
    o.ORDER_SOURCE,

    COALESCE(p.PAYMENT_METHOD, 'UNKNOWN') AS PAYMENT_METHOD,

    CASE
        WHEN o.ORDER_STATUS = 'CANCELLED' THEN 'NO_PAYMENT'
        ELSE COALESCE(p.PAYMENT_STATUS, 'NOT_AVAILABLE')
    END AS PAYMENT_STATUS,

    COALESCE(p.PAYMENT_GATEWAY, 'UNKNOWN') AS PAYMENT_GATEWAY,

    o.TOTAL_AMOUNT,
    o.DISCOUNT_AMOUNT,
    o.DELIVERY_FEE,
    o.TAX_AMOUNT,
    o.FINAL_AMOUNT,

    o.DELIVERY_DISTANCE_KM,
    o.ESTIMATED_DELIVERY_TIME,
    o.ACTUAL_DELIVERY_TIME,

    COALESCE(p.AMOUNT, 0) AS PAYMENT_AMOUNT,
    COALESCE(p.REFUND_AMOUNT, 0) AS REFUND_AMOUNT

FROM SILVER.ORDER_CLEAN o

LEFT JOIN SILVER.PAYMENT_CLEAN p
    ON o.ORDER_ID = p.ORDER_ID

INNER JOIN GOLD.DIM_CUSTOMER dc
    ON o.CUSTOMER_ID = dc.CUSTOMER_ID
   AND dc.IS_CURRENT = TRUE

INNER JOIN GOLD.DIM_RESTAURANT dr
    ON o.RESTAURANT_ID = dr.RESTAURANT_ID

LEFT JOIN GOLD.DIM_AGENT da
    ON o.AGENT_ID = da.AGENT_ID

LEFT JOIN GOLD.DIM_PROMOTION dp
    ON o.PROMO_CODE = dp.PROMO_CODE

INNER JOIN GOLD.DIM_DATE dd
    ON TO_NUMBER(TO_CHAR(CAST(o.ORDER_PLACED_AT AS DATE), 'YYYYMMDD')) = dd.DATE_SK;

In [ ]:
%%sql -r dataframe_18
SELECT *
FROM GOLD.FACT_ORDERS
LIMIT 20;

In [ ]:
%%sql -r dataframe_19
SELECT PROMO_CODE, COUNT(*)
FROM GOLD.DIM_PROMOTION
GROUP BY PROMO_CODE
HAVING COUNT(*) > 1;

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE TABLE GOLD.DIM_PROMOTION AS
SELECT

    ROW_NUMBER() OVER (ORDER BY PROMO_CODE) AS PROMOTION_SK,

    PROMO_CODE,

    MAX(PROMO_TYPE) AS PROMO_TYPE,
    MAX(DISCOUNT_VALUE) AS DISCOUNT_VALUE,
    MAX(MIN_ORDER_VALUE) AS MIN_ORDER_VALUE,
    MAX(START_DATE) AS START_DATE,
    MAX(END_DATE) AS END_DATE,
    MAX(IS_ACTIVE) AS IS_ACTIVE

FROM SILVER.PROMOTION_CLEAN
GROUP BY PROMO_CODE;

In [ ]:
%%sql -r dataframe_21
DROP TABLE GOLD.FACT_ORDERS;

In [ ]:
%%sql -r dataframe_22
CREATE OR REPLACE TABLE GOLD.FACT_ORDER_ITEMS AS
SELECT

    ORDER_ITEM_ID,
    ORDER_ID,
    ITEM_NAME,
    CATEGORY,
    IS_VEG,
    QUANTITY,
    UNIT_PRICE,
    TOTAL_PRICE

FROM SILVER.ORDER_ITEM_CLEAN;

In [ ]:
SELECT SOURCE_FILE_NAME, COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS
GROUP BY SOURCE_FILE_NAME;

In [ ]:
SELECT
ORDER_ID,
COUNT(*)
FROM SILVER.ORDER_CLEAN
GROUP BY ORDER_ID
HAVING COUNT(*) > 1;

In [ ]:
SELECT
ORDER_ID,
COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS
GROUP BY ORDER_ID
HAVING COUNT(*) > 1;

In [ ]:
SELECT COUNT(*) TOTAL_DUPLICATES
FROM (
    SELECT ORDER_ID
    FROM QUARANTINE.BAD_ORDER_RECORDS
    GROUP BY ORDER_ID
    HAVING COUNT(*) > 1
);

In [ ]:
CREATE OR REPLACE TABLE QUARANTINE.BAD_ORDER_RECORDS_CLEAN AS

SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY ORDER_ID
               ORDER BY INGESTION_TS DESC
           ) RN
    FROM QUARANTINE.BAD_ORDER_RECORDS
)
WHERE RN = 1;

In [ ]:
SELECT COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS_CLEAN;

In [ ]:
SELECT
COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS_CLEAN
WHERE ORDER_ID IS NULL;

In [ ]:
SELECT
SOURCE_FILE_NAME,
COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS_CLEAN
GROUP BY SOURCE_FILE_NAME
ORDER BY 2 DESC;

In [ ]:
%%sql -r dataframe_31
SELECT
COUNT(*)
FROM QUARANTINE.BAD_ORDER_RECORDS
WHERE ORDER_ID IS NULL;